# Lab 4 · Text classification: sort forum posts by topic
**Goal:** read a post and say which of 6 topics it belongs to; know which posts we can't classify.

**You will practise**
- Real text data: look at lengths, empties, examples
- Find a leak hidden in text (headers, signatures)
- Respect time: the test set is later than the training set
- Classic baseline: TF-IDF + logistic regression, tuned with CV
- Explain it: the words that drive each topic
- Pretrained sentence embeddings (MiniLM)
- Learn from very few labels
- Read mistakes, measure by text length, save and use

**How to work:** 🧠 = write your guess first · ✓ = the notebook proves the result

## 0 · Setup
- First run downloads the dataset (≈ 14 MB) and MiniLM (≈ 90 MB); later runs use the cache

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

SEED = 0
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})
TOPICS = ["comp.graphics", "rec.autos", "rec.sport.hockey", "sci.med", "sci.space", "talk.politics.guns"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 1 · Load the raw posts and look
- 20 Newsgroups: real forum posts from the 1990s; we use 6 topics
- The official test set was posted **later** than the training set

In [ ]:
def load(remove=()):
    kw = dict(categories=TOPICS, remove=remove, random_state=SEED)
    return fetch_20newsgroups(subset="train", **kw), fetch_20newsgroups(subset="test", **kw)

raw_train, raw_test = load()
names = raw_train.target_names
print(f"train {len(raw_train.data)} posts · test {len(raw_test.data)} posts · topics {names}")
print(pd.Series(raw_train.target).map(dict(enumerate(names))).value_counts().to_dict())
print("\n--- one raw post ---\n", raw_train.data[0][:700])

**What to notice**
- Each post starts with **headers**: sender, email, organisation, subject
- Many end with a **signature**, and quote earlier posts
- Classes are roughly balanced

## 2 · 🧠 Leak hunt
- Train TF-IDF + logistic regression on the raw posts, then on posts with headers, signatures and quotes removed
- Which scores higher? Which number would you believe?

In [ ]:
my_guess = {"raw": None, "cleaned": None}

In [ ]:
def tfidf_model(C=10):
    return make_pipeline(TfidfVectorizer(sublinear_tf=True, min_df=2), LogisticRegression(C=C, max_iter=3000))

train, test = load(remove=("headers", "footers", "quotes"))
scores = {}
for label, (tr, te) in {"raw": (raw_train, raw_test), "cleaned": (train, test)}.items():
    m = tfidf_model().fit(tr.data, tr.target)
    scores[label] = {"CV": cross_val_score(tfidf_model(), tr.data, tr.target, cv=cv).mean(), "test": m.score(te.data, te.target)}
pd.DataFrame(scores).T.round(3)

In [ ]:
assert scores["raw"]["test"] > scores["cleaned"]["test"] + 0.05
print("✓ headers and signatures inflate the score: they name the sender, not the topic")

**Why it's a leak**
- The same people post in the same groups: names, emails and signatures give the answer away
- A new post from a new person has none of that
- Always train on what the model will really see: the cleaned text

## 3 · Clean data: empties and lengths

In [ ]:
lengths = pd.Series([len(t.split()) for t in train.data])
empty_train = int((lengths == 0).sum()); empty_test = sum(len(t.split()) == 0 for t in test.data)
print(f"empty after cleaning: train {empty_train} · test {empty_test} · median words {lengths.median():.0f}")
lengths.clip(upper=600).plot.hist(bins=40); plt.xlabel("words per post (clipped at 600)"); plt.show()

keep = lengths.to_numpy() > 0                      # drop empty posts from TRAINING only
X_train = [t for t, k in zip(train.data, keep) if k]; y_train = train.target[keep]
X_test, y_test = test.data, test.target            # never drop from the test set: real life sends empties too
print(f"training on {len(X_train)} posts")

## 4 · Baseline: TF-IDF + logistic regression, tuned
- **TF-IDF:** one column per word; weight = how often in this post × how rare overall
- Tune with cross-validation on the training set only

In [ ]:
# TODO: GridSearchCV over a TfidfVectorizer + LogisticRegression pipeline
# grid: ngram_range (1,1)/(1,2), min_df 1/2, C 1/10/30 · cv=cv
# then tfidf = grid.best_estimator_, tfidf_cv = grid.best_score_, tfidf_test = tfidf.score(X_test, y_test)

**CV vs test**
- The test posts were written later: new names, new events, new words
- Expect the test score to sit a little **below** CV: that gap is drift, and it's normal

## 5 · Explain it: the words behind each topic

In [ ]:
vocab = np.array(tfidf.named_steps["tfidfvectorizer"].get_feature_names_out())
coef = tfidf.named_steps["logisticregression"].coef_
pd.DataFrame({names[k]: vocab[np.argsort(-coef[k])[:8]] for k in range(len(names))})

**Check**
- The top words make sense for each topic: the model learned the subject, not the senders
- If you saw names or email domains here, you'd still have a leak

## 6 · 🧠 Pretrained sentence embeddings
- MiniLM turns each post into 384 numbers that capture meaning
- Train only a logistic regression on top
- Predict: better or worse than TF-IDF on the test set?

In [ ]:
my_embed_guess = None   # "better" or "worse"?

In [ ]:
reader = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# TODO: E_train / E_test = reader.encode(..., batch_size=64, normalize_embeddings=True)
# TODO: head = LogisticRegression(C=1, max_iter=3000); embed_cv (CV on E_train), fit, embed_test

In [ ]:
assert embed_test > tfidf_test
print(f"✓ pretrained embeddings beat TF-IDF on later posts: {embed_test:.1%} vs {tfidf_test:.1%}")

**Why**
- TF-IDF only knows words seen in training
- MiniLM already knows that “engine” and “transmission”, or “orbit” and “launch”, belong together
- That knowledge carries over to posts with new words

## 7 · 🧠 Very few labels
- Labelling is expensive. Train each model on only 10, 30 and 100 posts per topic
- Which model suffers more?

In [ ]:
rng = np.random.default_rng(SEED)
rows = {}
for n in (10, 30, 100):
    idx = np.concatenate([rng.choice(np.where(y_train == c)[0], n, replace=False) for c in range(len(names))])
    a = tfidf_model().set_params(tfidfvectorizer__min_df=1).fit([X_train[i] for i in idx], y_train[idx]).score(X_test, y_test)
    b = LogisticRegression(C=10, max_iter=3000).fit(E_train[idx], y_train[idx]).score(E_test, y_test)
    rows[f"{n} per topic"] = {"TF-IDF": a, "MiniLM": b}
few = pd.DataFrame(rows).T
few.plot(marker="o", ylim=(0.4, 1)); plt.ylabel("test accuracy"); plt.show()
few.round(3)

In [ ]:
assert few.loc["10 per topic", "MiniLM"] - few.loc["10 per topic", "TF-IDF"] > 0.2
print(f"✓ with 10 labels per topic: MiniLM {few.loc['10 per topic', 'MiniLM']:.0%} vs TF-IDF {few.loc['10 per topic', 'TF-IDF']:.0%}")

## 8 · Read the mistakes
- Which topics get confused?
- Accuracy by post length: are short posts the problem?

In [ ]:
pred = head.predict(E_test); probs = head.predict_proba(E_test)
print(classification_report(y_test, pred, target_names=names, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=[n.split(".")[-1] for n in names], cmap="Blues", colorbar=False, xticks_rotation=45)
plt.show()

In [ ]:
words = np.array([len(t.split()) for t in X_test])
bins = pd.cut(words, [-1, 0, 10, 30, 100, 10_000], labels=["empty", "1–10", "11–30", "31–100", "100+"])
by_len = pd.DataFrame({"correct": pred == y_test, "length": bins}).groupby("length", observed=True)["correct"].agg(["mean", "size"])
by_len.round(3)

In [ ]:
assert by_len.loc["empty", "mean"] < 0.5 and by_len.loc["100+", "mean"] > 0.9
print("✓ long posts are easy; empty and very short posts are guesses")

## 9 · 🧠 A rule for posts we can't classify
- Send a post to a person if it has fewer than 5 words **or** the top probability is below 0.5
- Predict: what share goes to a person? Accuracy on the rest?

In [ ]:
to_person = (words < 5) | (probs.max(axis=1) < 0.5)
acc_rest = (pred == y_test)[~to_person].mean()
print(f"to a person: {to_person.mean():.1%} · accuracy on the rest: {acc_rest:.1%} (all: {(pred == y_test).mean():.1%})")
assert acc_rest > embed_test + 0.04
print("✓ routing the hardest posts to a person lifts accuracy on the rest by several points")

## 10 · Save and use
- Save the small head with joblib; record which embedding model it needs
- One function does the whole job: text → topic, probability, needs a person?

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump({"head": head, "embedder": "sentence-transformers/all-MiniLM-L6-v2", "topics": names}, "models/topic_classifier.joblib")

bundle = joblib.load("models/topic_classifier.joblib")
embedder = SentenceTransformer(bundle["embedder"])
def classify(text):
    p = bundle["head"].predict_proba(embedder.encode([text], normalize_embeddings=True))[0]
    k = int(p.argmax())
    return {"topic": bundle["topics"][k], "p": round(float(p[k]), 2), "needs_person": bool(len(text.split()) < 5 or p[k] < 0.5)}

for t in ["My car engine overheats on the highway. Should I replace the radiator or the thermostat?",
          "NASA plans another shuttle launch next month to repair the telescope.",
          "thanks!"]:
    print(classify(t), "←", t)

In [ ]:
assert classify("My car engine overheats on the highway. Should I replace the radiator or the thermostat?")["topic"] == "rec.autos"
assert classify("thanks!")["needs_person"]
print("✓ the saved classifier sorts new posts and flags ones it can't judge")

## 11 · Your turn
- Add `"comp.windows.x"` as a 7th topic: which topic does it get confused with?
- Try MiniLM + TF-IDF together (stack the two feature sets): does it beat both?
- Change the person rule to probability < 0.7: how many go to a person now?

In [ ]:
# TODO

## ✅ Checklist for every text-classification project
- [x] Read real examples; checked lengths and empty texts
- [x] Removed text that leaks the answer (headers, signatures, IDs)
- [x] Kept the time order: trained on older, tested on newer
- [x] Tuned a TF-IDF baseline with cross-validation
- [x] Checked the top words make sense
- [x] Tried pretrained embeddings, especially with few labels
- [x] Read mistakes by topic and by length
- [x] Routed short or unsure texts to a person
- [x] Saved the model with the name of the embedder it needs